In [21]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

In [22]:
torch.cuda.is_available()
df = pd.read_csv("C38_ready.csv")

features = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'days_offset']
targets = ['x_error', 'y_error', 'z_error']

X = df[features].values
y = df[targets].values

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
split_idx = int(len(df) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

df_test = df.iloc[split_idx:].reset_index(drop=True)

scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_test = scaler_y.transform(y_test)

# X_tensor = torch.tensor(X, dtype=torch.float32)
# y_tensor = torch.tensor(y, dtype=torch.float32)
# dataset = TensorDataset(X_tensor, y_tensor)

train_dataset = TensorDataset(  
    torch.tensor(X_train, dtype=torch.float32), 
    torch.tensor(y_train, dtype=torch.float32)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32), 
    torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)


In [23]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(7, 64),
            nn.GELU(),
            nn.Linear(64, 64),
            nn.GELU(),
            nn.Linear(64, 3)
        )

    def forward(self, x):
        return self.network(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model().to(device=device)

loss_fn = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

epochs = 1000
for epoch in range(epochs):
    model.train()
    ov_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        y_pred = model(batch_X)
        loss = loss_fn(y_pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ov_loss += loss.item() * batch_X.size(0)

    epoch_loss = ov_loss / len(train_dataset)

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

Epoch [100/1000], Loss: 0.0938
Epoch [200/1000], Loss: 0.0775
Epoch [300/1000], Loss: 0.0507
Epoch [400/1000], Loss: 0.0059
Epoch [500/1000], Loss: 0.0040
Epoch [600/1000], Loss: 0.0037
Epoch [700/1000], Loss: 0.0045
Epoch [800/1000], Loss: 0.0055
Epoch [900/1000], Loss: 0.0027
Epoch [1000/1000], Loss: 0.0060


In [ ]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        y_pred = model(batch_X)
        all_preds.append(y_pred.cpu().numpy())
        all_targets.append(batch_y.cpu().numpy())

preds_array = np.vstack(all_preds)
targets_array = np.vstack(all_targets)

preds_km = scaler_y.inverse_transform(preds_array)
targets_km = scaler_y.inverse_transform(targets_array)

err3d_model = np.linalg.norm(preds_km - targets_km, axis=1)
err3d_raw_sgp4 = np.linalg.norm(targets_km, axis=1)

df_test['Err3D_Model'] = err3d_model
df_test['Err3D_SGP4'] = err3d_raw_sgp4

print("\n=== PODSUMOWANIE OGÓLNE 3D [km] ===")
print(f"SGP4  -> Średnia: {err3d_raw_sgp4.mean():.3f} | Mediana: {np.median(err3d_raw_sgp4):.3f} | 90th perc (ogon): {np.percentile(err3d_raw_sgp4, 90):.3f}")
print(f"Model -> Średnia: {err3d_model.mean():.3f} | Mediana: {np.median(err3d_model):.3f} | 90th perc (ogon): {np.percentile(err3d_model, 90):.3f}")

print("\n=== BŁĄD 3D W ROZBICIU NA DAYS_OFFSET [km] ===")
summary = df_test.groupby('days_offset').apply(lambda g: pd.Series({
    'Liczba_Próbek': len(g),
    'Mediana_SGP4': g['Err3D_SGP4'].median(),
    'Mediana_Model': g['Err3D_Model'].median(),
    'Średnia_SGP4': g['Err3D_SGP4'].mean(),
    'Średnia_Model': g['Err3D_Model'].mean(),
    '90perc_SGP4': np.percentile(g['Err3D_SGP4'], 90),
    '90perc_Model': np.percentile(g['Err3D_Model'], 90)
}))

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(summary.round(3).to_string())


=== PODSUMOWANIE OGÓLNE 3D [km] ===
SGP4  -> Średnia: 6.159 | Mediana: 1.491 | 90th perc (ogon): 3.639
Model -> Średnia: 8.943 | Mediana: 3.009 | 90th perc (ogon): 11.827

=== BŁĄD 3D W ROZBICIU NA DAYS_OFFSET [km] ===
             Liczba_Próbek  Mediana_SGP4  Mediana_Model  Średnia_SGP4  Średnia_Model  90perc_SGP4  90perc_Model
days_offset                                                                                                    
0                    106.0         1.253          2.937         1.398          4.228        2.439         8.457
1                    107.0         1.117          2.895         3.103          5.929        2.518         9.069
3                    107.0         1.331          2.625         5.669          8.509        2.154        10.498
5                    108.0         2.937          3.886        14.346         16.987        4.605        15.743


C:\Users\kozak\AppData\Local\Temp\ipykernel_23748\1778276260.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = df_test.groupby('days_offset').apply(lambda g: pd.Series({


In [26]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Szybki test na obecnych danych
X_train, X_test, y_train, y_test = train_test_split(X, y[:, 0], test_size=0.2, shuffle=False) # Chronologicznie!

gbr = GradientBoostingRegressor(n_estimators=100, random_state=42)
gbr.fit(X_train, y_train)
preds = gbr.predict(X_test)

print("MAE Gradient Boosting dla osi X:", mean_absolute_error(y_test, preds))

MAE Gradient Boosting dla osi X: 7.189051826077655
